# 01 — Ingesting LAZ files into TileDB

This notebook covers the full ingestion workflow:
- single tile ingest
- manifest inspection
- batch parallel ingest
- filtering to specific LAS classes
- ingesting to S3-compatible object storage

**Paths** — replace `/path/to/...` with your actual file and directory paths.

## Setup

In [ ]:
import alsdb
from alsdb import ALSDatabase

alsdb.setup_logging()  # INFO-level logging to stderr

## Single tile ingest

`ALSDatabase` creates the TileDB array on first write. CRS, bounding box, and acquisition year are read from the LAZ header automatically — no configuration file required.

In [ ]:
db = ALSDatabase(storage_type="local", uri="/path/to/my_array")

db.ingest("/path/to/tile.laz")

In [ ]:
# Already-ingested tiles are skipped automatically (manifest tracking)
db.ingest("/path/to/tile.laz")  # no-op — safe to call again

## Inspect the manifest

The manifest records every ingested file: its path, year, CRS, bounding box, point count, and status.

In [ ]:
for entry in db.list_ingested():
    print(
        f"{entry['filename']:40s}  "
        f"year={entry['year']}  "
        f"n_points={entry['n_points']:>10,}  "
        f"status={entry['status']}"
    )

print("\nStored CRS:", db.stored_crs())

## Batch ingest — parallel, with consolidation

`ingest_many` processes a list of LAZ files in parallel using a `ThreadPoolExecutor`.  
Fragment consolidation is triggered every `consolidate_every` tiles to keep read performance healthy as the array grows.

In [ ]:
from pathlib import Path

paths = sorted(Path("/path/to/als_tiles/").glob("*.laz"))
print(f"Found {len(paths)} tiles")

db.ingest_many(
    paths,
    max_workers=8,  # parallel LAZ readers / TileDB writers
    consolidate_every=50,  # merge fragments every 50 tiles
)

## Filter to specific LAS classification codes

Storing only ground (2) and vegetation (3–5) points reduces array size and is sufficient for all forest structure products.

In [ ]:
db.ingest(
    "/path/to/tile.laz",
    classes=[2, 3, 4, 5],  # ASPRS classification codes
    overwrite=True,  # re-ingest even if already present in manifest
)

## Ingest to S3-compatible object storage

The API is identical for S3-backed arrays; only the constructor arguments change.  
You can load credentials from a boto3 session to avoid hard-coding them.

In [ ]:
import boto3

session = boto3.Session(profile_name="my-profile")
creds = session.get_credentials().get_frozen_credentials()

db_s3 = ALSDatabase(
    storage_type="s3",
    uri="s3://my-bucket/als_array",
    url="https://s3.example.com",  # endpoint URL — omit for AWS
    region="eu-central-1",
    credentials={
        "AccessKeyId": creds.access_key,
        "SecretAccessKey": creds.secret_key,
    },
)

db_s3.ingest("/path/to/tile.laz")